In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

In [12]:
base_path = Path('data/processed/base.parquet')
if not base_path.exists():
    base_path = Path('base.parquet')

base = pd.read_parquet(base_path)

In [13]:
base.head()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,temperature_min_c,dew_point_max_c,dew_point_min_c,humidity_max_pct,humidity_min_pct,humidity_pct,wind_gust_ms,wind_speed_ms,wind_direction_deg,station_count
0,NE,2022-01-01 00:00:00,10508.822,2022,1.165217,974.582143,974.605357,973.880357,1.620000,23.971429,...,23.894643,20.936000,20.372000,82.680000,78.760000,82.392157,4.003704,1.503704,97.753149,144
1,NE,2022-01-01 01:00:00,10598.654,2022,0.339130,976.091379,976.155172,975.739655,68.233333,23.655172,...,23.581034,21.175472,20.639623,85.471698,81.981132,84.711538,4.121818,1.352727,104.333533,144
2,NE,2022-01-01 02:00:00,10625.716,2022,0.343478,973.700000,973.919298,973.631579,123.000000,23.119298,...,23.068421,21.048980,20.646939,87.208333,84.653061,86.937500,3.680000,1.287273,101.001139,144
3,NE,2022-01-01 03:00:00,10583.828,2022,0.097872,975.218966,975.636207,975.194828,457.171429,23.053448,...,22.996552,21.134000,20.800000,88.163265,85.959184,87.816327,3.430909,1.145455,96.908462,144
4,NE,2022-01-01 04:00:00,10495.137,2022,0.114894,973.732143,974.223214,973.712500,306.800000,22.785714,...,22.700000,21.095652,20.769565,88.777778,86.911111,88.444444,3.075472,1.056604,106.756377,144


## Calendario 

In [14]:
base['month'] = base['timestamp'].dt.month
base['weekday'] = base['timestamp'].dt.day_name()
base['weekend'] = base['timestamp'].dt.dayofweek >= 5
base['hour'] = base['timestamp'].dt.hour

In [15]:
base['weekday_num'] = base['timestamp'].dt.weekday

In [16]:
%pip install holidays
import holidays
northeast_states = [
    'AL',
    'BA',
    'CE',
    'MA',
    'PB',
    'PE',
    'PI',
    'RN',
    'SE'
]

br_holidays = holidays.country_holidays('BR')

state_holidays = {
    state: holidays.country_holidays(
        'BR',
        subdiv=state
    )
    for state in northeast_states
}

base['is_holiday_national'] = (
    base['timestamp']
    .dt.date
    .isin(br_holidays)
    .astype(int)
)

def count_states_on_holiday(date):

    if date in br_holidays:
        return 0

    return sum(
        date in state_holidays[state]
        for state in northeast_states
    )


base['states_on_holiday'] = (
    base['timestamp']
    .dt.date
    .apply(count_states_on_holiday)
)

base.head()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,wind_speed_ms,wind_direction_deg,station_count,month,weekday,weekend,hour,weekday_num,is_holiday_national,states_on_holiday
0,NE,2022-01-01 00:00:00,10508.822,2022,1.165217,974.582143,974.605357,973.880357,1.620000,23.971429,...,1.503704,97.753149,144,1,Saturday,True,0,5,0,0
1,NE,2022-01-01 01:00:00,10598.654,2022,0.339130,976.091379,976.155172,975.739655,68.233333,23.655172,...,1.352727,104.333533,144,1,Saturday,True,1,5,0,0
2,NE,2022-01-01 02:00:00,10625.716,2022,0.343478,973.700000,973.919298,973.631579,123.000000,23.119298,...,1.287273,101.001139,144,1,Saturday,True,2,5,0,0
3,NE,2022-01-01 03:00:00,10583.828,2022,0.097872,975.218966,975.636207,975.194828,457.171429,23.053448,...,1.145455,96.908462,144,1,Saturday,True,3,5,0,0
4,NE,2022-01-01 04:00:00,10495.137,2022,0.114894,973.732143,974.223214,973.712500,306.800000,22.785714,...,1.056604,106.756377,144,1,Saturday,True,4,5,0,0


## Encoding cíclico

In [17]:
# Encoding cíclico da hora
base['hour_sin'] = np.sin(2 * np.pi * base['hour'] / 24)
base['hour_cos'] = np.cos(2 * np.pi * base['hour'] / 24)

# Encoding cíclico do dia da semana
base['weekday_sin'] = np.sin(2 * np.pi * base['weekday_num'] / 7)
base['weekday_cos'] = np.cos(2 * np.pi * base['weekday_num'] / 7)

# Encoding cíclico do mês
base['month_sin'] = np.sin(
    2 * np.pi * (base['month'] - 1) / 12
)
base['month_cos'] = np.cos(
    2 * np.pi * (base['month'] - 1) / 12
)

base.head()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,hour,weekday_num,is_holiday_national,states_on_holiday,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos
0,NE,2022-01-01 00:00:00,10508.822,2022,1.165217,974.582143,974.605357,973.880357,1.620000,23.971429,...,0,5,0,0,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
1,NE,2022-01-01 01:00:00,10598.654,2022,0.339130,976.091379,976.155172,975.739655,68.233333,23.655172,...,1,5,0,0,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0
2,NE,2022-01-01 02:00:00,10625.716,2022,0.343478,973.700000,973.919298,973.631579,123.000000,23.119298,...,2,5,0,0,0.500000,0.866025,-0.974928,-0.222521,0.0,1.0
3,NE,2022-01-01 03:00:00,10583.828,2022,0.097872,975.218966,975.636207,975.194828,457.171429,23.053448,...,3,5,0,0,0.707107,0.707107,-0.974928,-0.222521,0.0,1.0
4,NE,2022-01-01 04:00:00,10495.137,2022,0.114894,973.732143,974.223214,973.712500,306.800000,22.785714,...,4,5,0,0,0.866025,0.500000,-0.974928,-0.222521,0.0,1.0


## Lags

In [18]:
base['load_lag_1h'] = base['load_mw'].shift(1)
base['load_lag_2h'] = base['load_mw'].shift(2)
base['load_lag_3h'] = base['load_mw'].shift(3)
base['load_lag_24h'] = base['load_mw'].shift(24)
base['load_lag_48h'] = base['load_mw'].shift(48)
base['load_lag_168h'] = base['load_mw'].shift(168)

base['load_mean_3h'] = base['load_mw'].shift(1).rolling(3).mean()
base['load_mean_6h'] = base['load_mw'].shift(1).rolling(6).mean()
base['load_mean_24h'] = base['load_mw'].shift(1).rolling(24).mean()

base['load_std_3h'] = base['load_mw'].shift(1).rolling(3).std()
base['load_std_6h'] = base['load_mw'].shift(1).rolling(6).std()
base['load_std_24h'] = base['load_mw'].shift(1).rolling(24).std()

base.head()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,load_lag_3h,load_lag_24h,load_lag_48h,load_lag_168h,load_mean_3h,load_mean_6h,load_mean_24h,load_std_3h,load_std_6h,load_std_24h
0,NE,2022-01-01 00:00:00,10508.822,2022,1.165217,974.582143,974.605357,973.880357,1.620000,23.971429,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NE,2022-01-01 01:00:00,10598.654,2022,0.339130,976.091379,976.155172,975.739655,68.233333,23.655172,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NE,2022-01-01 02:00:00,10625.716,2022,0.343478,973.700000,973.919298,973.631579,123.000000,23.119298,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NE,2022-01-01 03:00:00,10583.828,2022,0.097872,975.218966,975.636207,975.194828,457.171429,23.053448,...,10508.822,NaN,NaN,NaN,10577.730667,NaN,NaN,61.191431,NaN,NaN
4,NE,2022-01-01 04:00:00,10495.137,2022,0.114894,973.732143,974.223214,973.712500,306.800000,22.785714,...,10598.654,NaN,NaN,NaN,10602.732667,NaN,NaN,21.239769,NaN,NaN


In [19]:
base['ramp_1h'] = base['load_mw'].diff(1).shift(1)

In [20]:
#Targets
base['target_load_1h'] = base['load_mw'].shift(-1)
base['target_load_6h'] = base['load_mw'].shift(-6)
base['target_load_24h'] = base['load_mw'].shift(-24)

In [21]:
base = base.dropna().reset_index(drop=True)
base.head()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,load_mean_3h,load_mean_6h,load_mean_24h,load_std_3h,load_std_6h,load_std_24h,ramp_1h,target_load_1h,target_load_6h,target_load_24h
0,NE,2022-01-08 00:00:00,11926.333,2022,0.520000,974.148148,974.203704,973.448148,47.800000,24.492593,...,12420.102000,12237.736833,11584.597667,134.929127,221.810606,583.661108,-244.169,11635.094,9793.342,11660.956
1,NE,2022-01-08 01:00:00,11635.094,2022,0.253333,973.319643,973.403571,972.975000,1.060000,24.185714,...,12233.397333,12222.878833,11583.075958,292.610669,241.764288,582.678831,-338.512,11402.214,9865.790,11290.670
2,NE,2022-01-08 02:00:00,11402.214,2022,0.238298,974.063793,974.348276,973.979310,16.883333,24.013793,...,11942.090667,12139.194667,11584.575750,315.171078,343.039821,582.768185,-291.239,11147.334,10122.028,11049.698
3,NE,2022-01-08 03:00:00,11147.334,2022,0.486957,973.198214,973.580357,973.164286,615.487500,23.726786,...,11654.547000,12037.324500,11585.088542,262.600449,459.007934,582.594832,-232.880,10927.972,10205.706,10924.900
4,NE,2022-01-08 04:00:00,10927.972,2022,0.195652,973.167273,973.587273,973.147273,426.871429,23.521818,...,11394.880667,11814.139000,11581.656042,243.962677,518.641087,585.017287,-254.880,10275.910,10321.871,10726.672


In [22]:
base.tail()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,load_mean_3h,load_mean_6h,load_mean_24h,load_std_3h,load_std_6h,load_std_24h,ramp_1h,target_load_1h,target_load_6h,target_load_24h
40699,NE,2026-08-30 19:00:00,14732.475,2026,0.021212,975.022973,975.251351,974.924324,1297.912329,31.492105,...,13593.778000,12371.095833,12931.276833,1245.447100,1555.554662,1580.179317,1089.576,14748.239,13415.721,15309.979
40700,NE,2026-08-30 20:00:00,14748.239,2026,0.209091,975.313514,975.333784,974.989189,589.445946,30.151316,...,14404.422333,12972.719167,12899.577292,614.416698,1670.015240,1533.490668,-52.709,14907.176,13135.991,15255.274
40701,NE,2026-08-30 21:00:00,14907.176,2026,0.080000,975.983562,976.000000,975.584932,107.198630,28.260000,...,14755.299333,13589.470833,12877.769458,27.054499,1490.593328,1501.682676,15.764,14852.579,12924.672,15686.557
40702,NE,2026-08-30 22:00:00,14852.579,2026,0.120635,976.952113,976.957746,976.387324,33.144444,26.986301,...,14795.963333,14194.870667,12862.479292,96.634977,1028.477317,1477.901832,158.937,14383.492,12865.152,15660.982
40703,NE,2026-08-30 23:00:00,14383.492,2026,0.022581,978.195652,978.198551,977.484058,48.388235,25.819718,...,14835.998000,14620.210167,12853.685417,80.755431,457.699677,1464.804863,-54.597,13805.914,12934.367,15149.486
